In [2]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [3]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [4]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [5]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 15

eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3168256
5376


NaiveModel()

In [ ]:
model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:

Learning Rate: 4.00e-04



|▌         | 6.2% (00:46) Evaluating model on training data... (999/1000) [2126/34215]:                   

## Model Analysis -------------------------

In [ ]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


In [ ]:
for key, features in [("NLL", StockGPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]),
                      ("MAE", StockGPT_cfg["target_features"]),
                      ("Z^2", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockGPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockGPT_params), format_num(linearModel_params), "0"], 
                                       features, key)

In [ ]:
return

import importlib
import setup
importlib.reload(setup)
from setup import PATH_RESULTS_RESIDUALS

store_result(PATH_RESULTS_RESIDUALS, process_result(stockGPT, gpt_losses, gpt_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS_RESIDUALS))

In [ ]:
from data_scrapper import scrape_data, get_all_tickers
from data_filler import fill_data
from data_preprocessor import preprocess_data
from setup import API_KEY, TIMEFRAME
import pandas_market_calendars as mcal

In [ ]:
return ""

from setup import ID
all_tickers = get_all_tickers("raw_data/all_tickers_trimmed_1_30", API_KEY)
scrape_data(API_KEY,
              f"raw_data/data_{ID}min_2026",
              250,
              all_tickers,
              mcal.get_calendar("NYSE").schedule("2026-01-01","2026-8-1").index,
              TIMEFRAME)
fill_data(f"raw_data/data_{ID}min_2026",
          f"filled_raw_data/data_{ID}min_2026",
          mcal.get_calendar("NYSE").schedule("2026-01-01","2026-8-1").index)
preprocess_data(f"filled_raw_data/data_{ID}min_2026",
                f"preprocessed_data/data_{ID}min_2026",
                mcal.get_calendar("NYSE").schedule("2026-01-01","2026-8-1").index, [0.75, 0.9])

In [ ]:
dls, train_norms = build_dataloaders("preprocessed_data/data_1min_2026")

In [ ]:
test_losses = test_model(dls["test"], stockGPT, device, eval_bs)
print(test_losses)